# audio speed detection with synthetic training data

This notebook contains my solution for the NEOAI 2026 audio speed detection competition.

The training data contains normal-speed speech only. The test set contains both normal and sped-up clips. The goal is to assign a higher score to clips that are more likely to have been sped up.

Public ROC AUC: around **0.90**

### a small clarification

The competition provided the task, data, metric, statistical context, and a starter baseline. I wrote and adapted the Kaggle code myself, including synthetic sample generation, acoustic feature extraction, model training, and submission.

I am currently studying statistics independently, so the terminology here reflects what the problem requires, not a claim that I already know everything.


## approach

I created a supervised classification problem from the normal training data:

1. Keep each original clip as class 0.
2. Create a time-stretched copy with a random speed factor between 1.5 and 2.0 and label it as class 1.
3. Extract acoustic features from both versions.
4. Standardize the features.
5. Train a logistic regression classifier.
6. Use the predicted probability for class 1 as the test score.

No positive labels, so the positive class had to be homemade. hohoho.


## 0. setup

The required packages are installed from the offline wheels included with the competition dataset.


In [ ]:
!pip install /kaggle/input/competitions/neoai-2026-day-2-audio/dataset/wheels/librosa-0.11.0-py3-none-any.whl \
             /kaggle/input/competitions/neoai-2026-day-2-audio/dataset/wheels/soundfile-0.13.1-py2.py3-none-any.whl \
             /kaggle/input/competitions/neoai-2026-day-2-audio/dataset/wheels/tiktoken-0.12.0-cp312-cp312-manylinux_2_28_x86_64.whl


## 1. imports and configuration

A fixed random seed makes the generated speed factors reproducible. Feature extraction is parallelized with `joblib`.


In [ ]:
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

INPUT_DIR = Path('/kaggle/input/competitions/neoai-2026-day-2-audio/dataset')
RANDOM_STATE = 42
SAMPLE_RATE = 12000
SPEED_MIN = 1.5
SPEED_MAX = 2.0


## 2. load the data

The train and test tables contain paths to seven-second mono audio clips. The paths are resolved relative to the competition dataset directory.


In [ ]:
train_df = pd.read_csv(INPUT_DIR / 'train.csv')
test_df = pd.read_csv(INPUT_DIR / 'test.csv')

train_df['audio_path'] = train_df['audio_path'].map(lambda path: str(INPUT_DIR / path))
test_df['audio_path'] = test_df['audio_path'].map(lambda path: str(INPUT_DIR / path))

print(f'train: {len(train_df):,} clips: {train_df.speaker_id.nunique()} speakers')
print(f'test:  {len(test_df):,} clips: {test_df.speaker_id.nunique()} speakers')
train_df.head()


## 3. extract audio features

Speeding up speech affects spectral energy, timing, cepstral coefficients, and duration. Each clip is represented with:

- spectral centroid, bandwidth, rolloff, and flatness
- RMS energy and zero-crossing rate
- eight MFCCs and their first-order deltas
- duration and waveform statistics

For each frame-level feature, I keep the mean, standard deviation, 10th percentile, and 90th percentile.


In [ ]:
def distribution_stats(values):
    values = np.asarray(values)
    return [
        values.mean(),
        values.std(),
        np.percentile(values, 10),
        np.percentile(values, 90),
    ]


def extract_features(path, speed=1.0):
    waveform, sample_rate = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    waveform = waveform.astype(np.float32)

    if speed != 1.0:
        waveform = librosa.effects.time_stretch(waveform, rate=speed)

    n_fft = 512
    hop_length = 256
    spectrum = np.abs(librosa.stft(waveform, n_fft=n_fft, hop_length=hop_length))
    power = spectrum ** 2

    centroid = librosa.feature.spectral_centroid(S=spectrum, sr=sample_rate)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=spectrum, sr=sample_rate)[0]
    rolloff = librosa.feature.spectral_rolloff(S=spectrum, sr=sample_rate)[0]
    flatness = librosa.feature.spectral_flatness(S=spectrum)[0]
    rms = librosa.feature.rms(S=spectrum, frame_length=n_fft, hop_length=hop_length)[0]
    zcr = librosa.feature.zero_crossing_rate(waveform, frame_length=n_fft, hop_length=hop_length)[0]

    mel = librosa.feature.melspectrogram(S=power, sr=sample_rate, n_mels=32)
    log_mel = librosa.power_to_db(mel + 1e-9)
    mfcc = librosa.feature.mfcc(S=log_mel, n_mfcc=8)
    delta_mfcc = librosa.feature.delta(mfcc)

    features = []
    for signal in [centroid, bandwidth, rolloff, flatness, rms, zcr]:
        features.extend(distribution_stats(signal))

    features.extend(mfcc.mean(axis=1))
    features.extend(mfcc.std(axis=1))
    features.extend(delta_mfcc.mean(axis=1))
    features.extend(delta_mfcc.std(axis=1))
    features.extend([
        len(waveform) / sample_rate,
        np.mean(np.abs(waveform)),
        np.std(waveform),
        np.percentile(np.abs(waveform), 95),
        np.max(np.abs(waveform)),
    ])

    return np.asarray(features, dtype=np.float32)


## 4. create synthetic training examples

Each original training clip is used twice. The unchanged version receives label 0, while a randomly sped-up version receives label 1. This produces a balanced training set without external data or pretrained models.


In [ ]:
train_paths = train_df['audio_path'].to_numpy()
test_paths = test_df['audio_path'].to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
speed_factors = rng.uniform(SPEED_MIN, SPEED_MAX, size=len(train_paths))


def extract_pair(path, speed):
    return extract_features(path), extract_features(path, speed)


pairs = Parallel(n_jobs=-1, backend='loky')(
    delayed(extract_pair)(path, speed)
    for path, speed in tqdm(
        zip(train_paths, speed_factors),
        total=len(train_paths),
        desc='building synthetic train',
    )
)

X_train = np.asarray([features for pair in pairs for features in pair])
y_train = np.tile([0, 1], len(pairs))

X_test = np.asarray(
    Parallel(n_jobs=-1, backend='loky')(
        delayed(extract_features)(path)
        for path in tqdm(test_paths, desc='extracting test')
    )
)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)


## 5. train the classifier

The features have different numerical scales, so they are standardized before training. Logistic regression then produces a probability for the sped-up class.


In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, C=2.0, random_state=RANDOM_STATE),
)

model.fit(X_train, y_train)
test_scores = model.predict_proba(X_test)[:, 1]

pd.Series(test_scores).describe()


## 6. create the submission

ROC AUC evaluates ranking, so the positive-class probability is submitted directly without applying a threshold.


In [ ]:
submission = pd.DataFrame({
    'audio_id': test_df['audio_id'],
    'score': test_scores,
})

submission.to_csv('submission.csv', index=False)

print(f'saved submission.csv: {len(submission):,} rows')
submission.head(10)


## result
The synthetic training transformation closely matched the change present in the test set. The acoustic features captured that change well enough for a linear classifier.
